In [1]:

import gc
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

SAVE_DIR   = '/srv/work'
IN_FEAT    = f'{SAVE_DIR}/te_features_v4.parquet'
OUT_FEAT   = f'{SAVE_DIR}/te_features_v5.parquet'

ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
ID_COLS    = ROUTE_KEYS + ['window']
TARGET_COL = 'label_ventana'
SPLIT_COL  = 'split'
EPS        = 1e-9

print(f'Entrada : {IN_FEAT}')
print(f'Salida  : {OUT_FEAT}')
schema = pq.read_schema(IN_FEAT)
print(f'Schema v4: {len(schema.names)} columnas')
n_lag336 = sum(1 for c in schema.names if c.startswith('lag336_'))
n_lag48  = sum(1 for c in schema.names if c.startswith('lag48_'))
print(f'  lag336_* a eliminar : {n_lag336}')
print(f'  lag48_*  (conservar): {n_lag48}')
print(f'  Nuevas features v5  : y_lag1/2/3, streak_length, d_lat/jitter/loss, latency_ratio_lag48  (8 total)')
print(f'  Features modelo esperado: {113 - n_lag336 + 8}  (~100)')

Entrada : /srv/work/te_features_v4.parquet
Salida  : /srv/work/te_features_v5.parquet
Schema v4: 122 columnas
  lag336_* a eliminar : 21
  lag48_*  (conservar): 21
  Nuevas features v5  : y_lag1/2/3, streak_length, d_lat/jitter/loss, latency_ratio_lag48  (8 total)
  Features modelo esperado: 100  (~100)


In [2]:

print('Cargando te_features_v4.parquet...')
df = pd.read_parquet(IN_FEAT)
print(f'v4 cargado: {df.shape}')

lag336_cols = [c for c in df.columns if c.startswith('lag336_')]
df = df.drop(columns=lag336_cols)
gc.collect()

print(f'\nEliminadas {len(lag336_cols)} columnas lag336_*:')
for c in sorted(lag336_cols):
    print(f'  - {c}')
print(f'\nDataset post-eliminacion: {df.shape}')

print('\nOrdenando por ruta + ventana...')
df = df.sort_values(ROUTE_KEYS + ['window']).reset_index(drop=True)
print(f'OK — shape: {df.shape}')
print(f'Periodo: {df["window"].min()} -> {df["window"].max()}')
print(f'Rutas unicas: {df[ROUTE_KEYS].drop_duplicates().shape[0]:,}')

Cargando te_features_v4.parquet...
v4 cargado: (3763917, 122)

Eliminadas 21 columnas lag336_*:
  - lag336_delta_mad_jitter
  - lag336_delta_mad_lat
  - lag336_delta_mad_loss
  - lag336_delta_p95_jitter
  - lag336_delta_p95_lat
  - lag336_delta_p95_loss
  - lag336_mad_jitter
  - lag336_mad_lat
  - lag336_mad_loss
  - lag336_mean_jitter
  - lag336_mean_lat
  - lag336_mean_loss
  - lag336_p50_jitter
  - lag336_p50_lat
  - lag336_p50_loss
  - lag336_p95_jitter
  - lag336_p95_lat
  - lag336_p95_loss
  - lag336_std_jitter
  - lag336_std_lat
  - lag336_std_loss

Dataset post-eliminacion: (3763917, 101)

Ordenando por ruta + ventana...
OK — shape: (3763917, 101)
Periodo: 2026-05-08 00:00:00+00:00 -> 2026-07-22 23:30:00+00:00
Rutas unicas: 1,035


In [3]:

print('Computando y_lag1 / y_lag2 / y_lag3...')

grp = df.groupby(ROUTE_KEYS)[TARGET_COL]

df['y_lag1'] = df[TARGET_COL].astype(float)

df['y_lag2'] = grp.shift(1).astype(float)

df['y_lag3'] = grp.shift(2).astype(float)

gc.collect()

labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}

print(f'y_lag1 NaN : {df["y_lag1"].isna().sum():,}  (esperado: 0)')
print(f'y_lag2 NaN : {df["y_lag2"].isna().sum():,}  (primera ventana por ruta)')
print(f'y_lag3 NaN : {df["y_lag3"].isna().sum():,}  (primeras 2 ventanas por ruta)')

print('\nDistribucion y_lag1 (= label_ventana en t):')
vc = df['y_lag1'].value_counts().sort_index()
for cls, cnt in vc.items():
    print(f'  {labels_map[int(cls)]:6s} ({int(cls)}): {cnt:10,}  ({100*cnt/len(df):.1f}%)')

print('\nVerificacion anti-leakage: y_lag1 == label_ventana (shift 0 = identidad)')
match = (df['y_lag1'] == df[TARGET_COL].astype(float)).all()
assert match, 'ERROR: y_lag1 != label_ventana'
print(f'  OK — identidad confirmada (sin leakage: t predice t+2, t < t+2)')

Computando y_lag1 / y_lag2 / y_lag3...
y_lag1 NaN : 0  (esperado: 0)
y_lag2 NaN : 1,035  (primera ventana por ruta)
y_lag3 NaN : 2,070  (primeras 2 ventanas por ruta)

Distribucion y_lag1 (= label_ventana en t):
  Normal (0):  3,306,491  (87.8%)
  Baja   (1):    304,928  (8.1%)
  Media  (2):     73,414  (2.0%)
  Alta   (3):     79,084  (2.1%)

Verificacion anti-leakage: y_lag1 == label_ventana (shift 0 = identidad)
  OK — identidad confirmada (sin leakage: t predice t+2, t < t+2)


In [4]:

print('Computando streak_length (vectorizado)...')

prev_label = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(1)
df['__not_same__'] = (df[TARGET_COL] != prev_label).astype(int)
del prev_label; gc.collect()

df['__cumchange__'] = df['__not_same__'].cumsum()

df['streak_length'] = (
    df.groupby(ROUTE_KEYS + ['__cumchange__'])
    .cumcount()
    .add(1)
    .astype(np.int32)
)

df.drop(columns=['__not_same__', '__cumchange__'], inplace=True)
gc.collect()

print(f'streak_length NaN : {df["streak_length"].isna().sum():,}  (esperado: 0)')
print(f'\nEstadisticas:')
desc = df['streak_length'].describe().round(2)
for k, v in desc.items():
    print(f'  {k:5s}: {v}')

print(f'\nDistribucion (primeros 10 valores):')
vc_streak = df['streak_length'].value_counts().sort_index().head(10)
for v, cnt in vc_streak.items():
    print(f'  streak={v:3d}: {cnt:10,}  ({100*cnt/len(df):.2f}%)')
pct_gt10 = (df['streak_length'] > 10).mean() * 100
print(f'  streak>10  : {(df["streak_length"] > 10).sum():10,}  ({pct_gt10:.2f}%)')
print(f'  max streak : {df["streak_length"].max()}')

Computando streak_length (vectorizado)...
streak_length NaN : 0  (esperado: 0)

Estadisticas:
  count: 3763917.0
  mean : 115.18
  std  : 217.88
  min  : 1.0
  25%  : 7.0
  50%  : 36.0
  75%  : 125.0
  max  : 2932.0

Distribucion (primeros 10 valores):
  streak=  1:    355,404  (9.44%)
  streak=  2:    169,988  (4.52%)
  streak=  3:    117,867  (3.13%)
  streak=  4:     93,263  (2.48%)
  streak=  5:     78,886  (2.10%)
  streak=  6:     69,073  (1.84%)
  streak=  7:     61,983  (1.65%)
  streak=  8:     56,535  (1.50%)
  streak=  9:     52,340  (1.39%)
  streak= 10:     48,999  (1.30%)
  streak>10  :  2,659,579  (70.66%)
  max streak : 2932


In [5]:

df['d_latency'] = df['mean_lat']    - df['lag1_mean_lat']
df['d_jitter']  = df['mean_jitter'] - df['lag1_mean_jitter']
df['d_loss']    = df['mean_loss']   - df['lag1_mean_loss']

print('Primera diferencia de metricas:')
for col in ['d_latency', 'd_jitter', 'd_loss']:
    n_nan = df[col].isna().sum()
    print(f'  {col:15s}: media={df[col].mean():.4f}  std={df[col].std():.4f}  '
          f'NaN={n_nan:,}  ({100*n_nan/len(df):.1f}%)')

ref_nan = df['lag1_mean_lat'].isna().sum()
print(f'\nNaN esperado = {ref_nan:,} (primera ventana por ruta, mismo que lag1_mean_lat)')
print(f'\nInterpretacion directa:')
pct_lat_emp = (df['d_latency'] > 0).sum() * 100 / len(df)
pct_lat_mej = (df['d_latency'] < 0).sum() * 100 / len(df)
print(f'  d_latency > 0 (empeorando): {pct_lat_emp:.1f}%  |  < 0 (mejorando): {pct_lat_mej:.1f}%')
pct_loss_emp = (df['d_loss'] > 0).sum() * 100 / len(df)
print(f'  d_loss > 0    (empeorando): {pct_loss_emp:.1f}%')

Primera diferencia de metricas:
  d_latency      : media=-0.0000  std=0.0139  NaN=2,565  (0.1%)
  d_jitter       : media=0.0000  std=0.9686  NaN=2,568  (0.1%)
  d_loss         : media=0.0003  std=1.8058  NaN=2,447  (0.1%)

NaN esperado = 2,509 (primera ventana por ruta, mismo que lag1_mean_lat)

Interpretacion directa:
  d_latency > 0 (empeorando): 49.9%  |  < 0 (mejorando): 50.0%
  d_loss > 0    (empeorando): 26.3%


In [6]:

df['latency_ratio_lag48'] = df['mean_lat'] / (df['lag48_mean_lat'].abs() + EPS)

n_nan = df['latency_ratio_lag48'].isna().sum()
ref_nan_48 = df['lag48_mean_lat'].isna().sum()
print('latency_ratio_lag48:')
print(f'  NaN       : {n_nan:,}  ({100*n_nan/len(df):.1f}%)  (primeras 24h por ruta)')
print(f'  NaN ref   : {ref_nan_48:,}  (lag48_mean_lat, mismos NaN)')
valid = df['latency_ratio_lag48'].dropna()
print(f'  Mediana   : {valid.median():.4f}  (esperado ~1.0 — estabilidad relativa)')
print(f'  Percentil 1%  : {valid.quantile(0.01):.4f}')
print(f'  Percentil 99% : {valid.quantile(0.99):.4f}')
n_gt2 = (valid > 2.0).sum()
n_lt05 = (valid < 0.5).sum()
print(f'  Ratio > 2.0   : {n_gt2:,}  ({100*n_gt2/len(valid):.2f}%)  [latencia 2x vs 24h atras]')
print(f'  Ratio < 0.5   : {n_lt05:,}  ({100*n_lt05/len(valid):.2f}%)  [latencia 50% vs 24h atras]')

latency_ratio_lag48:
  NaN       : 58,145  (1.5%)  (primeras 24h por ruta)
  NaN ref   : 58,079  (lag48_mean_lat, mismos NaN)
  Mediana   : 0.9999  (esperado ~1.0 — estabilidad relativa)
  Percentil 1%  : 0.6539
  Percentil 99% : 1.5316
  Ratio > 2.0   : 16,241  (0.44%)  [latencia 2x vs 24h atras]
  Ratio < 0.5   : 15,968  (0.43%)  [latencia 50% vs 24h atras]


In [7]:

META_COLS   = ID_COLS + [TARGET_COL, SPLIT_COL]
N_SAMP_COLS = [c for c in df.columns if c.startswith('n_samp')]
feat_cols_v5 = [c for c in df.columns
                if c not in META_COLS
                and not c.startswith('n_samp')]

print('=== Feature Engineering v5 — Resumen ===')
print(f'Dataset shape    : {df.shape}')
print(f'Total columnas   : {len(df.columns)}')
print(f'  Meta           : {len(META_COLS)}  {META_COLS}')
print(f'  n_samp cols    : {len(N_SAMP_COLS)}  {N_SAMP_COLS}')
print(f'  FEATURES MODELO: {len(feat_cols_v5)}')
print()

n_lag336 = sum(1 for c in df.columns if c.startswith('lag336_'))
assert n_lag336 == 0, f'ERROR: {n_lag336} columnas lag336_* no eliminadas'
print(f'[OK] lag336_* eliminados : {n_lag336}  (D12-REV confirmado)')

new_feats_v5 = ['y_lag1', 'y_lag2', 'y_lag3', 'streak_length',
                'd_latency', 'd_jitter', 'd_loss', 'latency_ratio_lag48']
print(f'\n[OK] Nuevas features v5 ({len(new_feats_v5)}):')
for f in new_feats_v5:
    assert f in df.columns, f'ERROR: falta {f}'
    n_nan = df[f].isna().sum()
    pct_nan = 100 * n_nan / len(df)
    print(f'  {f:25s}: NaN={n_nan:>8,}  ({pct_nan:5.2f}%)')

print(f'\n=== Splits ===')
for split in ['train', 'val', 'test']:
    n = (df[SPLIT_COL] == split).sum()
    print(f'  {split:5s}: {n:>10,}')

print(f'\n=== Distribucion labels ===')
labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
dist = df[TARGET_COL].value_counts(normalize=True).sort_index() * 100
for k, pct in dist.items():
    print(f'  {labels_map[k]:6s} ({k}): {pct:.1f}%')

print(f'\n=== Composicion de features ({len(feat_cols_v5)}) ===')
n_lag1   = sum(1 for c in feat_cols_v5 if c.startswith('lag1_'))
n_lag2   = sum(1 for c in feat_cols_v5 if c.startswith('lag2_'))
n_lag48  = sum(1 for c in feat_cols_v5 if c.startswith('lag48_'))
n_y_lag  = sum(1 for c in feat_cols_v5 if c.startswith('y_lag'))
n_temp   = sum(1 for c in feat_cols_v5 if c in ['hour_sin','hour_cos','dow_sin','dow_cos','is_weekend'])
n_bgp    = sum(1 for c in feat_cols_v5 if c.startswith('bgp_'))
n_dmet   = sum(1 for c in feat_cols_v5 if c.startswith('d_'))
n_ratio  = sum(1 for c in feat_cols_v5 if c == 'latency_ratio_lag48')
n_streak = sum(1 for c in feat_cols_v5 if c == 'streak_length')
n_base   = len(feat_cols_v5) - n_lag1 - n_lag2 - n_lag48 - n_y_lag - n_temp - n_bgp - n_dmet - n_ratio - n_streak
print(f'  Base de red (p50/p95/mad/mean/std/delta): {n_base}')
print(f'  Lag t-1                                 : {n_lag1}')
print(f'  Lag t-2                                 : {n_lag2}')
print(f'  Lag t-48 (24h)                          : {n_lag48}')
print(f'  y_lag (autoregresion etiquetas) [NUEVO] : {n_y_lag}')
print(f'  streak_length                   [NUEVO] : {n_streak}')
print(f'  d_metric (primera diferencia)   [NUEVO] : {n_dmet}')
print(f'  latency_ratio_lag48             [NUEVO] : {n_ratio}')
print(f'  HOD+DOW temporal                        : {n_temp}')
print(f'  BGP                                     : {n_bgp}')
print(f'  ─────────────────────────────────────────')
print(f'  TOTAL MODELO                            : {len(feat_cols_v5)}')

print(f'\n=== Nulos en nuevas features ===')
nulls = df[new_feats_v5].isnull().sum()
nulls = nulls[nulls > 0]
if not nulls.empty:
    print(nulls.sort_values(ascending=False).to_string())
else:
    print('  Sin nulos en nuevas features (excepto los ya reportados arriba)')

=== Feature Engineering v5 — Resumen ===
Dataset shape    : (3763917, 109)
Total columnas   : 109
  Meta           : 6  ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']
  n_samp cols    : 3  ['n_samp_jitter', 'n_samp_lat', 'n_samp_loss']
  FEATURES MODELO: 100

[OK] lag336_* eliminados : 0  (D12-REV confirmado)

[OK] Nuevas features v5 (8):
  y_lag1                   : NaN=       0  ( 0.00%)
  y_lag2                   : NaN=   1,035  ( 0.03%)
  y_lag3                   : NaN=   2,070  ( 0.05%)
  streak_length            : NaN=       0  ( 0.00%)
  d_latency                : NaN=   2,565  ( 0.07%)
  d_jitter                 : NaN=   2,568  ( 0.07%)
  d_loss                   : NaN=   2,447  ( 0.07%)
  latency_ratio_lag48      : NaN=  58,145  ( 1.54%)

=== Splits ===
  train:  2,125,778
  val  :    644,690
  test :    993,449

=== Distribucion labels ===
  Normal (0): 87.8%
  Baja   (1): 8.1%
  Media  (2): 2.0%
  Alta   (3): 2.1%

=== Composicion de feature

In [8]:

print('Guardando te_features_v5.parquet...')
df.to_parquet(OUT_FEAT, index=False)

print(f'\n✅ Guardado: {OUT_FEAT}')
print(f'   Shape    : {df.shape}')
print(f'   Features : {len(feat_cols_v5)}  (para modelo)')
print(f'   Splits   : {df[SPLIT_COL].value_counts().to_dict()}')

print(f'\n=== Cambios v4 -> v5 (resumen) ===')
print(f'  ELIMINADAS : 21 cols lag336_*  (D12-REV, Strobl et al. 2007)')
print(f'  AGREGADAS  :  8 features nuevas')
print(f'    y_lag1/2/3       (Box & Jenkins 1976; Widiawati et al. 2025)')
print(f'    streak_length    (Rabiner 1989)')
print(f'    d_lat/jitter/loss(Holt 1957/2004)')
print(f'    latency_ratio_lag48 (Barford et al. 2002)')
print(f'  RESULTADO  : {113 - 21} - 21 + 8 = {113 - 21 + 8} -> {len(feat_cols_v5)} features modelo')
print(f'\nSiguiente: 05 modelado v5.ipynb')
print(f'  Criterio H2 (anti-HARKing, Kerr 1998; Benchmarking v3 final 2026-08-01):')
print(f'    QWK >= 0.65  AND  RecAlta >= 0.55  (simultaneamente, en validacion primero)')

Guardando te_features_v5.parquet...

✅ Guardado: /srv/work/te_features_v5.parquet
   Shape    : (3763917, 109)
   Features : 100  (para modelo)
   Splits   : {'train': 2125778, 'test': 993449, 'val': 644690}

=== Cambios v4 -> v5 (resumen) ===
  ELIMINADAS : 21 cols lag336_*  (D12-REV, Strobl et al. 2007)
  AGREGADAS  :  8 features nuevas
    y_lag1/2/3       (Box & Jenkins 1976; Widiawati et al. 2025)
    streak_length    (Rabiner 1989)
    d_lat/jitter/loss(Holt 1957/2004)
    latency_ratio_lag48 (Barford et al. 2002)
  RESULTADO  : 92 - 21 + 8 = 100 -> 100 features modelo

Siguiente: 05 modelado v5.ipynb
  Criterio H2 (anti-HARKing, Kerr 1998; Benchmarking v3 final 2026-08-01):
    QWK >= 0.65  AND  RecAlta >= 0.55  (simultaneamente, en validacion primero)
